In [ ]:
import os
import time
from binance.client import Client
from binance.enums import SIDE_BUY, SIDE_SELL, ORDER_TYPE_MARKET
import pandas as pd
import numpy as np

In [ ]:
API_KEY =  'Your Public Key'
API_SECRET = 'Your Secret Key'
TESTNET = True  # set to False when you're ready for real trading

client = Client(API_KEY, API_SECRET, testnet=TESTNET)

In [ ]:
def place_order(side, quantity):
    SYMBOL = "BTCUSDT"
    try:
        order = client.create_order(
            symbol=SYMBOL,
            side=side,
            type=ORDER_TYPE_MARKET,
            quantity=round(quantity, 5)  # Binance requires max 5 decimal places
        )
        print(f"Order placed: {side} | Quantity: {quantity}")
        return order
    except Exception as e:
        print(f"Order failed: {e}")
        return None
    
def get_data():
    SYMBOL = "BTCUSDT"
    INTERVAL = Client.KLINE_INTERVAL_1HOUR
    klines = client.get_klines(symbol=SYMBOL, interval=INTERVAL)
    df = pd.DataFrame(klines, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ])
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")
    df[["open", "high", "low", "close", "volume"]] = df[["open", "high", "low", "close", "volume"]].astype(float)
    return df

def generate_signal(df):
    df["EMA_50"] = df["close"].ewm(span=50, adjust=False).mean()
    df["EMA_200"] = df["close"].ewm(span=200, adjust=False).mean()
    df["signal"] = 0
    df.loc[df["EMA_50"] > df["EMA_200"], "signal"] = 1
    df.loc[df["EMA_50"] < df["EMA_200"], "signal"] = -1
    return df

def rebalance_position(df):
    df["log_returns"] = np.log(df["close"] / df["close"].shift(1))
    target_volatility = 0.0001
    #daily volatility estimation using rolling window 
    df["volatility"] = df["log_returns"].rolling(window=24).std()
    df["volatility"].fillna(method="bfill", inplace=True)
    
    #position sizing based on inverse volatility
    df["position_size"] = target_volatility / df["volatility"]
    df["weight_rebalanced"] = df.loc[df["open_time"].dt.hour == 0, "position_size"]
    df["weight_rebalanced"] = df["weight_rebalanced"].fillna(method="bfill")
    return df

def check_signal(df, capital):
    global in_position_trend

    btc_price = df["close"].iloc[-1]
    trend_capital = capital * 0.5      # 50% for trend
    rebalance_capital = capital * 0.5  # 50% for rebalance

    # --- TREND PORTION (50% capital) ---
    # Signal = 1 means buy, signal = -1 means sell
    signal_now = df["signal"].iloc[-1]

    if signal_now == 1 and not in_position_trend:
        trend_qty = round(trend_capital / btc_price, 6)
        print(f"TREND BUY | Qty: {trend_qty}")
        order = place_order(SIDE_BUY, trend_qty)
        if order:
            in_position_trend = True

    elif signal_now == -1 and in_position_trend:
        trend_qty = round(trend_capital / btc_price, 6)
        print(f"TREND SELL | Qty: {trend_qty}")
        order = place_order(SIDE_SELL, trend_qty)
        if order:
            in_position_trend = False

    # --- REBALANCE PORTION (50% capital) ---
    # Buy/sell the DIFFERENCE in weight, not the full weight
    weight_diff = df["weight_rebalanced"].iloc[-1] - df["weight_rebalanced"].iloc[-2]

    if weight_diff > 0:
        rebalance_qty = round((weight_diff * rebalance_capital) / btc_price, 6)
        print(f"REBALANCE BUY | Qty: {rebalance_qty} | rebalance_weight: {df['weight_rebalanced'].iloc[-1]}")
        order = place_order(SIDE_BUY, rebalance_qty)

    elif weight_diff < 0:
        rebalance_qty = round((abs(weight_diff) * rebalance_capital) / btc_price, 6)
        print(f"REBALANCE SELL | Qty: {rebalance_qty} | rebalance_weight: {df['weight_rebalanced'].iloc[-1]}")
        order = place_order(SIDE_SELL, rebalance_qty)

    else:
        print("No rebalance needed.")

def main():
    global in_position_trend
    in_position_trend = False
    CAPITAL = 1000

    while True:
        try:
            df = get_data()
            df = generate_signal(df)
            df = rebalance_position(df)
            check_signal(df, CAPITAL)
            time.sleep(900)  # wait for 15 minutes before checking again
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(60)
if __name__ == "__main__":
    main()
